# 앙상블 비교 — MPDD 시뮬레이션 vs 실제 보호소 데이터

모델 쌍마다 "둘 중 하나라도 top-K 안에서 맞히면 성공"(OR 커버리지)을 계산해서, 앙상블했을 때
이론적으로 얼마나 좋아질 수 있는지 어림. (예전엔 `scripts/compare_ensembles.py` 로 따로 뺐었는데,
그룹을 계속 추가할 예정이라 여기 노트북 하나에 다 넣어둠 — 밑에 `GROUPS` 딕셔너리에 항목만 추가하면 됨.)

각 표 아래 "단독 성능"도 같이 보여줘서, 앙상블이 단일 모델 대비 얼마나 더 나아지는지 바로 비교 가능.

In [1]:
from itertools import combinations
from pathlib import Path

import pandas as pd

DERIVED = Path("../dataset/derived")

GROUPS = {
    "MPDD_hard_corrupt (시뮬레이션)": {
        "CLIP-ReID": DERIVED / "clipreid_case_errors.csv",
        "MegaDescriptor": DERIVED / "megadescriptor_case_errors.csv",
        "PetFace": DERIVED / "petface_case_errors.csv",
        "ARBase": DERIVED / "arbase_case_errors.csv",
    },
    "shelter_hard (진짜 보호소, 개+고양이 섞임)": {
        "CLIP-ReID": DERIVED / "shelter_clipreid_errors.csv",
        "MegaDescriptor": DERIVED / "shelter_megadescriptor_errors.csv",
        "PetFace": DERIVED / "shelter_petface_errors.csv",
        "ARBase": DERIVED / "shelter_arbase_errors.csv",
    },
    "shelter_hard_dogs (진짜 보호소, 개만)": {
        "CLIP-ReID": DERIVED / "dogs_clipreid_errors.csv",
        "MegaDescriptor": DERIVED / "dogs_megadescriptor_errors.csv",
        "PetFace": DERIVED / "dogs_petface_errors.csv",
        "ARBase": DERIVED / "dogs_arbase_errors.csv",
    },
    # "shelter_hard_cats (진짜 보호소, 고양이만)": {
    #     "CLIP-ReID": DERIVED / "cats_clipreid_errors.csv",
    #     ...
    # },
}
TOPK = [1, 5]


def or_coverage(ranks_a, ranks_b, pids, k):
    """pids 중 A 또는 B 가 rank<=k 로 맞힌 비율 (rank<=0 은 gallery에 아예 없어서 못 맞힌 것)."""
    hit = sum(1 for pid in pids if (0 < ranks_a[pid] <= k) or (0 < ranks_b[pid] <= k))
    return hit / len(pids)


def load_ranks(model_to_csv):
    ranks, missing = {}, []
    for model, csv_path in model_to_csv.items():
        csv_path = Path(csv_path)
        if not csv_path.exists():
            missing.append((model, csv_path))
            continue
        df = pd.read_csv(csv_path)
        ranks[model] = dict(zip(df["case_pid"], df["rank"]))
    return ranks, missing


def show_group(group_name, model_to_csv):
    ranks, missing = load_ranks(model_to_csv)
    for model, path in missing:
        print(f"[건너뜀] {model}: {path} 없음")
    if len(ranks) < 2:
        print(f"=== {group_name} === 비교할 모델이 2개 미만이라 건너뜀")
        return

    pid_sets = [set(r.keys()) for r in ranks.values()]
    pids = set.intersection(*pid_sets)

    rows = {
        f"{a} + {b}": {f"Top-{k}": or_coverage(ranks[a], ranks[b], pids, k) for k in TOPK}
        for a, b in combinations(ranks, 2)
    }
    table = pd.DataFrame(rows).T

    print(f"=== {group_name} (n={len(pids)}) ===")
    display(table.style.format("{:.1%}").background_gradient(cmap="Greens", axis=None))

    solo = pd.Series(
        {name: sum(1 for pid in pids if r[pid] == 1) / len(pids) for name, r in ranks.items()},
        name="단독 Recall@1",
    )
    display(solo.map("{:.1%}".format))

## 그룹별로 실행 — 필요한 만큼 셀 복사해서 이름만 바꾸면 됨

In [2]:
for group_name, model_to_csv in GROUPS.items():
    show_group(group_name, model_to_csv)

=== MPDD_hard_corrupt (시뮬레이션) (n=96) ===


,Top-1,Top-5
CLIP-ReID + MegaDescriptor,78.1%,85.4%
CLIP-ReID + PetFace,77.1%,85.4%
CLIP-ReID + ARBase,75.0%,90.6%
MegaDescriptor + PetFace,85.4%,90.6%
MegaDescriptor + ARBase,79.2%,91.7%
PetFace + ARBase,79.2%,91.7%


CLIP-ReID         60.4%
MegaDescriptor    72.9%
PetFace           65.6%
ARBase            66.7%
Name: 단독 Recall@1, dtype: str

=== shelter_hard (진짜 보호소, 개+고양이 섞임) (n=344) ===


,Top-1,Top-5
CLIP-ReID + MegaDescriptor,86.6%,95.9%
CLIP-ReID + PetFace,89.0%,98.0%
CLIP-ReID + ARBase,84.0%,96.8%
MegaDescriptor + PetFace,87.5%,95.3%
MegaDescriptor + ARBase,83.4%,94.2%
PetFace + ARBase,86.0%,96.2%


CLIP-ReID         77.6%
MegaDescriptor    75.0%
PetFace           78.5%
ARBase            66.9%
Name: 단독 Recall@1, dtype: str

=== shelter_hard_dogs (진짜 보호소, 개만) (n=199) ===


,Top-1,Top-5
CLIP-ReID + MegaDescriptor,90.5%,96.0%
CLIP-ReID + PetFace,94.0%,98.5%
CLIP-ReID + ARBase,88.9%,97.0%
MegaDescriptor + PetFace,92.5%,97.0%
MegaDescriptor + ARBase,88.4%,95.0%
PetFace + ARBase,92.0%,97.5%


CLIP-ReID         83.9%
MegaDescriptor    80.4%
PetFace           86.9%
ARBase            72.9%
Name: 단독 Recall@1, dtype: str